# Sweep Playground

Interactive exploration of sweep figures — single-variable parameter
sweeps that produce a metric-vs-axis curve per algorithm.

**Compatible experiments:** `snr_sweep`, `gamma_sweep`, `kappa_sweep`,
`clutter_cnr_sweep`, `n_ue_sweep`, `n_ap_sweep`, `antennas_sweep`
(all use `kind == 'sweep'` and the `plot_sweep` function).

In [ ]:
# Stage 12: shared setup — make `cordis` importable when this notebook
# is launched from notebooks/, then apply the IEEE paper rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style, load_latest_result, load_run, summarize,
)

import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH (e.g. on a compute node).
setup_paper_style(use_latex=True)

logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

In [ ]:
# ── The one knob: which sweep to load ──
EXPERIMENT = 'n_ue_sweep'       # any 'kind=sweep' experiment

result = load_latest_result(EXPERIMENT)
assert result.kind == 'sweep', (
    f'This notebook is for sweep experiments; got kind={result.kind}.'
)
summarize(result)

## 1. Quick sweep plot — one metric, all algorithms

`plot_sweep` takes the keyed-by-x results dict (its x-axis comes
from the dict's keys, not from `sweep_axis`).  The `sweep_axis`
metadata is used for human-readable axis labels — set them yourself
via `ax.set_xlabel(result.sweep_axis.display)` etc.

In [ ]:
from cordis.plotting import plot_sweep, figsize

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=ax)
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('min-SINR [dB]')
ax.set_title(f'{EXPERIMENT}: min-SINR vs. {result.sweep_axis.display}')
plt.show()

## 2. Side-by-side: SINR + SCNR

Most paper figures pair a communication metric with a sensing metric.
Build the two-panel layout inline — no helper needed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=figsize(width='double', aspect=2.5/1.5))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=axes[0])
axes[0].set_xlabel(result.sweep_axis.display)
axes[0].set_ylabel('min-SINR [dB]')
axes[0].set_title('min-SINR [dB]')
plot_sweep(result.sweep_results, metric='mean_scnr_db', ax=axes[1])
axes[1].set_xlabel(result.sweep_axis.display)
axes[1].set_ylabel('mean-SCNR [dB]')
axes[1].set_title('mean-SCNR [dB]')
plt.tight_layout()
plt.show()

## 3. Algorithm filter — focus on CORDIS vs. Centralized

`plot_sweep` honors the same `only=` keyword as `plot_cdf`.

In [ ]:
PICK = ['CORDIS-Split', 'CORDIS-ADMM', 'Centralized']

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=ax, only=PICK)
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('mean-SINR [dB]')
plt.show()

## 4. Log-y for SCNR sweeps

Sensing metrics often span orders of magnitude across the sweep range.

In [ ]:
fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='mean_scnr_db', ax=ax, only=PICK)
ax.set_yscale('log')
ax.set_xlabel(result.sweep_axis.display)
ax.set_ylabel('mean SCNR (log scale)')
plt.show()

## 5. Annotate the operating point

If your paper highlights a specific axis value (e.g. SNR=10 dB), draw
a vertical line + label at that point.

In [ ]:
x_marker = 130.0                # adjust to your sweep's axis units

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_sweep(result.sweep_results, metric='min_sinr_db', ax=ax, only=PICK)
ax.set_xlabel(result.sweep_axis.display)
if min(result.sweep_axis.values) <= x_marker <= max(result.sweep_axis.values):
    ax.axvline(x_marker, color='gray', linestyle=':', linewidth=1)
    ax.annotate(f'  operating point ({x_marker})',
                xy=(x_marker, ax.get_ylim()[1]),
                xytext=(5, -10), textcoords='offset points',
                fontsize=8, color='gray')
plt.show()

## 6. Figsize variants

In [ ]:
# Figsize variants — `figsize` returns (w, h) in inches for matplotlib.
# width: 'single' (one column), 'double' (two-column), 'third' (3-up panel).
# aspect: w/h ratio.  Tweak both to fit your paper layout.
from cordis.plotting import figsize

for width in ('single', 'double', 'third'):
    w, h = figsize(width=width, aspect=3/2)
    print(f'{width:>6}: ({w:.2f}, {h:.2f}) inches')

# Example: tight three-up panel for a paper sub-figure
# fig, axes = plt.subplots(1, 3, figsize=figsize(width='double', aspect=3.5/1.5))

## 7. Save with provenance metadata

In [ ]:
# Save with provenance metadata (Git SHA, creation date, etc. — embedded
# into the PDF's metadata, prepended as comments in the .pgf).
from cordis.plotting import save_figure

# Adjust EXPERIMENT and metric labels to match the figure above.
out = save_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_demo',
    formats=('pdf', 'png'),       # add 'pgf' on systems with LaTeX
    metadata={'Experiment': EXPERIMENT, 'Notebook': 'playground'},
)
for p in out:
    print('wrote', p)